In [1]:
from troma import (
    CombinatorialProblem,
    ConstraintSketchMap,
    matching_pursuit,
    get_optimizer,
    bind_optimizer
)

import numpy as np
from matplotlib import pyplot as plt
from functools import partial
from qamomile.optimization.qaoa import QAOAConverter

In [2]:
from problems_generator import compressible_opt_pb as co_pb
#Define problem
number_spins = 12
rules_reward = {(0, 0, 0, 0): 18, (1, 1, 0, 1): 5, (1, 1, 1, 1): 18}

objective = partial(co_pb.estimate_cost, rules_rewards=rules_reward)

In [3]:
# 1. Create problem
problem = CombinatorialProblem(objective, problem_size=12, problem_dimension=2)

# 2. Sample
samp = problem.sampling(n_samples=800, seed=3)
# print(samp)

# 3. Create sketch map and sketch
sketch_map = ConstraintSketchMap(sketch_length=12, interaction_size=2, constraints="nearest_neighbors")
problem_sketch = problem.sketching(sketch_map)

ham = problem_sketch.to_hamiltonian()
ham

Hamiltonian(terms={(0,): -245.25, (1,): 370.5, (0, 1): 339.75, (2,): 27.5, (1, 2): 985.75, (3,): -536.5, (2, 3): 1127.25, (4,): -557.5, (3, 4): 1197.75, (5,): -851.5, (4, 5): 1381.25, (6,): -381.5, (5, 6): 1462.25, (7,): 30.5, (6, 7): 998.25, (8,): -643.5, (7, 8): 1175.25, (9,): -117.5, (8, 9): 1351.25, (10,): -681.5, (9, 10): 975.25, (11,): 144.25, (10, 11): 565.25}, num_qubits=12)

In [4]:
from functools import reduce
from operator import mul
from qamomile.optimization.binary_model.expr import spin
from qamomile.optimization.binary_model.model import BinaryModel
from qamomile.optimization.binary_model.expr import VarType

ising_model = ham.to_ising_model()
ising_model.coefficients
hubo_model = ham.to_hubo()

In [5]:
converter = QAOAConverter(hubo_model)
converter.spin_model = converter.spin_model.normalize_by_abs_max()
hamiltonian = converter.get_cost_hamiltonian()
print(hamiltonian)

Hamiltonian((Z0,): -0.16772097794494786, (Z1,): 0.2533766455804411, (Z2,): 0.01880663361258335, (Z3,): -0.3669003248418533, (Z4,): -0.3812617541460079, (Z5,): -0.5823217644041716, (Z6,): -0.26089929902547443, (Z7,): 0.020858266370319713, (Z8,): -0.44007522653445036, (Z9,): -0.08035561634467431, (Z10,): -0.46606257479911095, (Z11,): 0.09864934176782356, (Z0, Z1): 0.23234740981364335, (Z1, Z2): 0.674132330312874, (Z2, Z3): 0.7709010087194392, (Z3, Z4): 0.8191143785262438, (Z4, Z5): 0.9446059155411182, (Z5, Z6): 1.0, (Z6, Z7): 0.6826808001367756, (Z7, Z8): 0.8037271328432211, (Z8, Z9): 0.9240895879637545, (Z9, Z10): 0.6669516156607967, (Z10, Z11): 0.38656180543682683)


In [6]:
from qamomile.qiskit import QiskitExecutor
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager


class IBMRuntimeExecutor(QiskitExecutor):
    """QiskitExecutor that runs sampling through a pre-built SamplerV2."""

    def __init__(self, sampler, backend, estimator=None, optimization_level=1):
        """
        Args:
            sampler: A configured SamplerV2 instance (e.g. SamplerV2(mode=backend),
                     SamplerV2(mode=Session(backend=...)), or SamplerV2(mode=Batch(...))).
            backend: The Qiskit backend the sampler is targeting. Used to build
                     the ISA pass manager so circuits are transpiled to the
                     backend's native gate set before being sent to the sampler.
            estimator: Optional EstimatorV2 for expectation values.
            optimization_level: Preset pass manager optimization level (0-3).
        """
        super().__init__(backend=backend, estimator=estimator)
        self._sampler = sampler
        self._pm = generate_preset_pass_manager(
            optimization_level=optimization_level, backend=backend
        )

    def execute(self, circuit, shots):
        circuit = self._ensure_measurements(circuit)
        isa_circuit = self._pm.run(circuit)

        job = self._sampler.run([isa_circuit], shots=shots)
        pub_result = job.result()[0]

        # SamplerV2 stores counts under the classical register name(s).
        # measure_all() -> "meas"; explicit named registers -> that name.
        data = pub_result.data
        reg_name = next(iter(data))
        return getattr(data, reg_name).get_counts()
    
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2, QiskitRuntimeService
backend = AerSimulator(seed_simulator=901, max_parallel_threads=1)
sampler = SamplerV2(mode=backend, options={"default_shots": 5048})
my_executor = IBMRuntimeExecutor(sampler, backend)

In [7]:
from qamomile.qiskit import QiskitTranspiler
transpiler = QiskitTranspiler()
p=5
executable = converter.transpile(transpiler, p=p)

In [12]:
import os

from qiskit_ibm_runtime import SamplerV2
import numpy as np
from scipy.optimize import minimize

# Seed the simulator so re-executing the notebook reproduces the same
# COBYLA trajectory and final sampling distribution. Without a seed,
# every shot draws fresh randomness, COBYLA sees a noisy cost surface,
# and each notebook run converges to a different (but equivalent) local
# optimum.

docs_test_mode = os.environ.get("QAMOMILE_DOCS_TEST") == "1"
sample_shots = 256 if docs_test_mode else 2048
maxiter = 25 if docs_test_mode else 1000

rng = np.random.default_rng(900)
initial_params = rng.uniform(0, np.pi, 2 * p)

cost_history = []


def cost_fn(params):
    gammas = list(params[:p])
    betas = list(params[p:])
    job = executable.sample(
        my_executor,
        shots=sample_shots,
        bindings={"gammas": gammas, "betas": betas},
    )
    result = job.result()
    # decode_to_binary_sampleset returns the QUBO-domain BinarySampleSet
    # whose `energy` is the penalized objective — what COBYLA needs to
    # see infeasibility costs. The polymorphic decode() returns an
    # ommx.v1.SampleSet whose `objective` is the un-penalized true
    # objective; using it here would let the optimizer settle on
    # infeasible all-zero / all-one bitstrings.
    decoded = converter.decode_to_binary_sampleset(result)
    energy = - decoded.energy_mean()
    cost_history.append(energy)
    return energy


res = minimize(
    cost_fn,
    initial_params,
    method="COBYLA",
    options={"maxiter": maxiter},
)

print(f"Optimized cost: {res.fun:.3f}")
print(f"Optimal params: {[round(v, 4) for v in res.x]}")
print(f"Function evaluations: {res.nfev}")

Optimized cost: -3.824
Optimal params: [np.float64(0.1511), np.float64(0.1001), np.float64(2.9965), np.float64(0.3624), np.float64(2.3914), np.float64(2.6525), np.float64(3.6049), np.float64(0.5621), np.float64(1.8136), np.float64(1.3923)]
Function evaluations: 88


In [ ]:
gammas_opt = list(res.x[:p])
betas_opt = list(res.x[p:])
sample_result = executable.sample(
    my_executor,
    shots=1000,
    bindings={"gammas": gammas_opt, "betas": betas_opt},
).result()

sample_set = converter.decode(sample_result)

In [23]:
from troma import DitString
max_idx = sample_set.energy.index(max(sample_set.energy))
best = sample_set.samples[max_idx]
best_bitstring = best.values()
print(best_bitstring)
DitString(best_bitstring).to_integer('L') # -> should be 4095 or 0

dict_values([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])


4095

In [18]:
# 4. Run matching pursuit
result = matching_pursuit(problem_sketch, iteration_number=5,optimizer=get_optimizer("spin_chain_nn_max"))

# 5. Get results
print(result.positions)      # Selected line positions

[4095    0 2730 1365 2530]


In [22]:
backend = AerSimulator()
sampler = SamplerV2(mode=backend)

opti = bind_optimizer("qaoa", sampler=sampler,number_shots=4096)
result = matching_pursuit(problem_sketch, iteration_number=5, step=None, optimizer=opti)
print(result.positions)      # Selected line positions

[4095 2048  682 1365  482]
